# YOLOv8 QAT Training + TensorRT INT8 Export

## Buoc 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
from pathlib import Path

BASE_DIR   = Path("/content")
OUTPUT_DIR = BASE_DIR / "dataset"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DRY_RUN = False  # False = xóa thật

total_orphan_labels = 0
total_orphan_images = 0

for split in ["train", "val", "test"]:
    img_dir = OUTPUT_DIR / split / "images"
    lbl_dir = OUTPUT_DIR / split / "labels"

    img_files  = {f.stem: f for f in img_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS}
    lbl_stems  = {f.stem for f in lbl_dir.iterdir() if f.suffix == ".txt"}

    orphan_labels = lbl_stems - img_files.keys()
    orphan_images = img_files.keys() - lbl_stems

    print(f"[{split}]  orphan labels: {len(orphan_labels):4d} | orphan images: {len(orphan_images):4d}")

    if not DRY_RUN:
        for stem in orphan_labels:
            (lbl_dir / f"{stem}.txt").unlink()
        for stem in orphan_images:
            img_files[stem].unlink()

    total_orphan_labels += len(orphan_labels)
    total_orphan_images += len(orphan_images)

print(f"\nTổng đã xóa — labels: {total_orphan_labels} | images: {total_orphan_images}")

# Kiểm tra lại
print("\nKết quả sau khi xóa:")
for split in ["train", "val", "test"]:
    imgs = len([f for f in (OUTPUT_DIR / split / "images").iterdir() if f.suffix.lower() in IMAGE_EXTS])
    lbls = len(list((OUTPUT_DIR / split / "labels").glob("*.txt")))
    status = "OK" if imgs == lbls else "LỆCH"
    print(f"  {split:6s} → images: {imgs:6d} | labels: {lbls:6d}  [{status}]")


[train]  orphan labels:    0 | orphan images:    0
[val]  orphan labels:    0 | orphan images:    0
[test]  orphan labels:    0 | orphan images:    0

Tổng đã xóa — labels: 0 | images: 0

Kết quả sau khi xóa:
  train  → images:   3636 | labels:   3636  [OK]
  val    → images:   2434 | labels:   2434  [OK]
  test   → images:   2012 | labels:   2012  [OK]


## Buoc 2: Clone repo va setup thu muc

In [ ]:
import os

WORKDIR = "/content/drive/MyDrive/yolov8-qat"
REPO    = f"{WORKDIR}/yolov8-QAT"

!mkdir -p {WORKDIR}
%cd {WORKDIR}

if not os.path.exists(REPO):
    !git clone https://github.com/mmsori/yolov8-QAT.git
else:
    print("Repo da ton tai, bo qua clone.")

%cd {REPO}

## Buoc 3: Kiem tra moi truong (bat buoc co GPU)

In [ ]:
import torch, subprocess, sys

# ── Thong tin GPU ─────────────────────────────────────────────
assert torch.cuda.is_available(), "Khong co GPU! Vao Runtime > Change runtime type > GPU"

gpu_name  = torch.cuda.get_device_name(0)
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
cuda_ver  = torch.version.cuda
sm        = torch.cuda.get_device_capability(0)

print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA       : {cuda_ver}")
print(f"GPU        : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"SM version : {sm[0]}.{sm[1]}")

# ── Kiem tra RAM he thong (High RAM = ~52GB, Standard = ~12GB) ─
try:
    mem_gb = int(open("/proc/meminfo").read().split("MemTotal:")[1].split()[0]) / 1e6
    print(f"RAM he thong: {mem_gb:.1f} GB")
except:
    mem_gb = 0

# ── Ket luan ──────────────────────────────────────────────────
print()
if "A100" in gpu_name and vram_gb > 70:
    print("[OK] A100 80GB (SXM/PCIe)")
elif "A100" in gpu_name and vram_gb > 35:
    print("[OK] A100 40GB")
elif "A100" in gpu_name:
    print("[OK] A100 (VRAM thap hon mong doi, co the shared memory)")
elif "T4" in gpu_name:
    print("[INFO] T4 16GB — du chay INT8 benchmark")
elif "V100" in gpu_name:
    print("[INFO] V100 — khong co INT8 Tensor Core, bench INT8 se chậm")
else:
    print(f"[WARN] GPU khac: {gpu_name}")

if mem_gb > 40:
    print("[OK] High RAM (>40GB)")
elif mem_gb > 0:
    print(f"[INFO] Standard RAM ({mem_gb:.0f}GB) — du de train/export")


Python     : 3.12.12
PyTorch    : 2.9.0+cu126
CUDA       : 12.6
GPU        : NVIDIA A100-SXM4-80GB
VRAM       : 85.1 GB
SM version : 8.0
RAM he thong: 175.2 GB

[OK] A100 80GB (SXM/PCIe)
[OK] High RAM (>40GB)


## Buoc 4: Cai dat packages

> **Chi chay 1 lan** sau khi clone repo.

In [7]:
# 4a. Cai custom ultralytics (yolov8-QAT fork)
# KHONG dung "pip install ultralytics" - se ghi de custom code!
%cd /content/drive/MyDrive/QAT/yolov8-QAT
!pip install -e . -q

/content/drive/MyDrive/QAT/yolov8-QAT
  Preparing metadata (setup.py) ... done


In [8]:
# 4b. Cai pytorch_quantization cua NVIDIA
!pip install --upgrade setuptools wheel -q
!pip install nvidia-pyindex -q
!pip install --no-cache-dir pytorch-quantization==2.1.2 --extra-index-url https://pypi.ngc.nvidia.com -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 189.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 7.6 MB/s eta 0:00:00


In [9]:
# 4c. Fix numpy (pytorch_quantization can numpy < 2.0)
!pip install "numpy<2.0" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 73.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompa

In [1]:
# 4d. Kiem tra compatibility PyTorch 2.9 + pytorch_quantization
import torch
import pytorch_quantization, ultralytics
from pytorch_quantization import quant_modules, nn as quant_nn

print(f"pytorch_quantization : {pytorch_quantization.__version__}")
print(f"ultralytics (custom) : {ultralytics.__version__}")
print(f"PyTorch              : {torch.__version__}")

# Test 1: quant_modules co patch duoc Conv2d khong
quant_modules.initialize()
test_conv = torch.nn.Conv2d(3, 16, 3)
assert "QuantConv2d" in type(test_conv).__name__, "FAIL: Conv2d khong duoc patch thanh QuantConv2d"
print("[OK] quant_modules.initialize() hoat dong")

# Test 2: TensorQuantizer hoat dong
from pytorch_quantization.tensor_quant import QuantDescriptor
q = quant_nn.TensorQuantizer(QuantDescriptor(num_bits=8))
x = torch.randn(1, 3, 64, 64).cuda()
q.cuda()(x)
print("[OK] TensorQuantizer hoat dong tren GPU")

# Test 3: GradScaler (deprecated warning trong PyTorch 2.x nhung van chay)
try:
    scaler = torch.cuda.amp.GradScaler(enabled=False)
    print("[OK] GradScaler hoat dong (co the co deprecation warning, binh thuong)")
except Exception as e:
    print(f"[WARN] GradScaler loi: {e}")

print()
print("==> Moi thu san sang, co the chay QAT!")


pytorch_quantization : 2.1.2
ultralytics (custom) : 8.0.228
PyTorch              : 2.10.0+cu128
[OK] quant_modules.initialize() hoat dong
[OK] TensorQuantizer hoat dong tren GPU
[OK] GradScaler hoat dong (co the co deprecation warning, binh thuong)

==> Moi thu san sang, co the chay QAT!


/tmp/ipykernel_53601/4284648840.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=False)


## Buoc 5: Chuan bi dataset

In [ ]:
# Copy dataset tu Drive ve RAM Colab (nhanh hon khi train)
!cp -r "/content/drive/MyDrive/yolov8s_trash/dataset" "/content/dataset"
!ls /content/dataset

In [2]:
import yaml

data_yaml = {
    "train": "/content/drive/MyDrive/yolov8/dataset/train",
    "val":   "/content/drive/MyDrive/yolov8/dataset/val",
    "test":  "/content/drive/MyDrive/yolov8/dataset/test",
    "nc":    3,
    "names": ["PAPER", "PLASTIC", "GLASS"],
}
with open("/content/data.yaml", "w") as f:
    yaml.dump(data_yaml, f, allow_unicode=True)
print("data.yaml tao xong:")
!cat /content/data.yaml

data.yaml tao xong:
names:
- PAPER
- PLASTIC
- GLASS
nc: 3
test: /content/drive/MyDrive/yolov8/dataset/test
train: /content/drive/MyDrive/yolov8/dataset/train
val: /content/drive/MyDrive/yolov8/dataset/val


## Buoc 6: Chay QAT Training

In [ ]:
%cd /content/drive/MyDrive/QAT/yolov8-QAT

!python qat_nvidia.py \
    --model-config     yolov8s.yaml \
    --pretrained-weight /content/drive/MyDrive/QAT/phase_C.pt \
    --data-config      /content/data.yaml \
    --epochs           10 \
    --imgsz            640 \
    --batch            64 \
    --device           0 \
    --workers          16 \
    --cache            ram \
    --optimizer        AdamW \
    --lr0              1e-3 \
    --lrf              0.1 \
    --momentum         0.90 \
    --weight-decay     0.0004 \
    --freeze           10 \
    --box              0.04 \
    --cls              2.5 \
    --dfl              1.2 \
    --mosaic           0.0 \
    --patience         5 \
    --project          qat \
    --name             train \
    --plots


/content/drive/MyDrive/QAT/yolov8-QAT
Loading custom weight: /content/drive/MyDrive/QAT/phase_C.pt
Ultralytics YOLOv8.0.228 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.yaml, data=/content/data.yaml, epochs=10, time=None, patience=5, batch=64, imgsz=640, save=True, save_period=-1, cache=ram, device=0, workers=16, project=qat, name=train, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=10, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, show=False, save_frames=False, save_txt=False, save_

TRUST RESULT

In [4]:
%cd /content/drive/MyDrive/QAT/yolov8-QAT

!python qat_nvidia.py \
  --pretrained-weight /content/drive/MyDrive/QAT/phase_C.pt \
  --data-config /content/data.yaml \
  --model-config yolov8s.yaml \
  --epochs 10 \
  --batch 16 \
  --freeze 6 \
  --optimizer AdamW \
  --lr0 1e-2 \
  --weight-decay 5e-4 \
  --momentum 0.937 \
  --box 7.5 --cls 0.5 --dfl 1.5 \
  --mosaic 0.0 --mixup 0.0 --fliplr 0.5 \
  --recalib-every 1 \
  --patience 0 \
  --project qat \
  --name entropy_skip_detect




/content/drive/MyDrive/QAT/yolov8-QAT
Loading custom weight: /content/drive/MyDrive/QAT/phase_C.pt
Ultralytics YOLOv8.0.228 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.yaml, data=/content/data.yaml, epochs=10, time=None, patience=0, batch=16, imgsz=640, save=True, save_period=-1, cache=ram, device=0, workers=8, project=qat, name=entropy_skip_detect, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=6, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, show=False, save_frames=False, save_txt=

## Buoc 7: Export QAT -> ONNX -> TensorRT INT8 Engine

> **Luu y**: Khong dung `model.export(format="engine", int8=True)` vi do la PTQ.

In [5]:
# Cai them onnx va tensorrt
!pip install onnx onnxsim -q
!pip install --extra-index-url https://pypi.nvidia.com tensorrt==10.0.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 GB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 246.6 MB/s eta 0:00:00


In [7]:
!pip install onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 19.7 MB/s eta 0:00:00


In [ ]:
# khong chay cell nay

%%writefile /content/drive/MyDrive/yolov8-qat/yolov8-QAT/export_engine.py
"""Export QAT YOLOv8: .pt -> ONNX with Q/DQ nodes -> TensorRT INT8 Engine"""
import argparse, sys
from pathlib import Path
import torch
from pytorch_quantization import nn as quant_nn, quant_modules
from ultralytics.nn.modules import Detect, RTDETRDecoder
from ultralytics.nn.tasks import DetectionModel
from ultralytics.qat.nvidia_tensorrt.quant_ops import quant_module_change


def export_onnx(cfg, weights, nc, imgsz=640):
    quant_nn.TensorQuantizer.use_fb_fake_quant = True
    quant_modules.initialize()
    model = DetectionModel(cfg=cfg, ch=3, nc=nc, verbose=False)
    quant_module_change(model)
    ckpt = torch.load(weights, map_location="cpu")
    if isinstance(ckpt, dict) and ("model" in ckpt or "ema" in ckpt):
        src = ckpt.get("ema") or ckpt["model"]
        state_dict = src.float().state_dict()
        src_label = "ema" if ckpt.get("ema") is not None else "model"
        print(f"[INFO] Loaded checkpoint [{src_label}] from {weights}")
    else:
        state_dict = ckpt
        print(f"[INFO] Loaded state dict: {weights}")
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    for m in model.modules():
        if isinstance(m, (Detect, RTDETRDecoder)):
            m.export = True
            m.format = "onnx"
    dummy = torch.randn(1, 3, imgsz, imgsz)
    onnx_path = Path(weights).with_suffix(".tensorrt.onnx")
    torch.onnx.export(
        model, dummy, str(onnx_path),
        verbose=False, opset_version=17,
        input_names=["images"], output_names=["output0"],
        dynamic_axes={"images": {0: "batch"}, "output0": {0: "batch"}},
    )
    print(f"[OK] ONNX saved: {onnx_path}")
    return onnx_path


def export_engine_python(onnx_path, imgsz=640, workspace=4, fp16=False):
    try:
        import tensorrt as trt
    except ImportError:
        print("[ERROR] Cai tensorrt: pip install tensorrt")
        sys.exit(1)
    logger = trt.Logger(trt.Logger.INFO)
    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(f"[Parse Error] {parser.get_error(i)}")
            sys.exit(1)
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, workspace * (1 << 30))
    config.set_flag(trt.BuilderFlag.INT8)
    if fp16:
        config.set_flag(trt.BuilderFlag.FP16)
    profile = builder.create_optimization_profile()
    shape = (1, 3, imgsz, imgsz)
    profile.set_shape("images", shape, shape, shape)
    config.add_optimization_profile(profile)
    print("[INFO] Building TensorRT engine (co the mat 5-15 phut)...")
    engine_bytes = builder.build_serialized_network(network, config)
    if engine_bytes is None:
        print("[ERROR] Build engine that bai!")
        sys.exit(1)
    engine_path = Path(onnx_path).with_suffix(".engine")
    with open(engine_path, "wb") as f:
        f.write(engine_bytes)
    print(f"[OK] TensorRT engine saved: {engine_path}")
    return engine_path


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-config", type=str, default="yolov8s.yaml")
    parser.add_argument("--weight", type=str, required=True)
    parser.add_argument("--nc", type=int, required=True)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--workspace", type=int, default=4)
    parser.add_argument("--fp16", action="store_true")
    parser.add_argument("--onnx-only", action="store_true")
    args = parser.parse_args()
    onnx_path = export_onnx(args.model_config, args.weight, args.nc, args.imgsz)
    if not args.onnx_only:
        export_engine_python(onnx_path, args.imgsz, args.workspace, args.fp16)


In [10]:
REPO = "/content/drive/MyDrive/QAT/yolov8-QAT"
%cd {REPO}

BEST_PT = "/content/drive/MyDrive/QAT/yolov8-QAT/qat/entropy_skip_detect/weights/best.pt"

!python export_engine.py \
    --model-config yolov8s.yaml \
    --weight {BEST_PT} \
    --nc 3 \
    --imgsz 640 \
    --workspace 4

# Output:
# qat/entropy_skip_detect/weights/best.tensorrt.onnx
# qat/entropy_skip_detect/weights/best.engine


# Output:
# runs/detect/train4/weights/best.tensorrt.onnx
# runs/detect/train4/weights/best.engine

/content/drive/MyDrive/QAT/yolov8-QAT
Overriding model.yaml nc=80 with nc=3
W0324 06:30:51.139977 134047147975808 tensor_quantizer.py:281] Use Pytorch's native experimental fake quantization.
Add C2fQuantChunk to model.2
Add QuantAdd to model.2.m.0
Add C2fQuantChunk to model.4
Add QuantAdd to model.4.m.0
Add QuantAdd to model.4.m.1
Add C2fQuantChunk to model.6
Add QuantAdd to model.6.m.0
Add QuantAdd to model.6.m.1
Add C2fQuantChunk to model.8
Add QuantAdd to model.8.m.0
Add QuantUpsample to model.10
None
2.0
nearest
Add QuantConcat to model.11
Add C2fQuantChunk to model.12
Add QuantUpsample to model.13
None
2.0
nearest
Add QuantConcat to model.14
Add C2fQuantChunk to model.15
Add QuantConcat to model.17
Add C2fQuantChunk to model.18
Add QuantConcat to model.20
Add C2fQuantChunk to model.21
[INFO] Loaded checkpoint [model] from /content/drive/MyDrive/QAT/yolov8-QAT/qat/entropy_skip_detect/weights/best.pt
[INFO] Skipped 38 quantizers in Detect head (model.22) → FP32
/content/drive/MyDri

## Buoc 9: Benchmark — mAP + Inference Speed (FP32 vs INT8)

| Model | mAP50 | mAP50-95 | Latency (ms/img) | FPS | size |
|---|---|---|---|---|---|
| FP32 (goc) | 0.8903 | 0.7865 | 9.21ms | 108.6 | 22.5MB |
| INT8 engine | 0.8857 | 0.7876 | 3.96ms | 252.7 | 19.1MB |

> Chay cell duoi de dien vao bang tren.

In [11]:
import json
from pathlib import Path

ENGINE_PATH = "/content/drive/MyDrive/QAT/yolov8-QAT/qat/entropy_skip_detect/weights/best.tensorrt.engine"

with open(ENGINE_PATH, "rb") as f:
    raw = f.read()

# Tìm pure engine bytes (bỏ metadata cũ nếu có)
def strip_metadata(data):
    try:
        meta_len = int.from_bytes(data[:4], byteorder="little")
        if 0 < meta_len < 65536:
            parsed = json.loads(data[4:4+meta_len].decode("utf-8"))
            if isinstance(parsed, dict) and "task" in parsed:
                print(f"  Found existing metadata ({meta_len}B), stripping...")
                return data[4+meta_len:]
    except Exception:
        pass
    return data  # raw engine bytes

engine_bytes = strip_metadata(raw)

# Ghi lại với đúng format ultralytics
metadata = {
    "description": "YOLOv8 QAT INT8",
    "stride": 32,
    "task": "detect",
    "batch": 1,
    "imgsz": [640, 640],
    "names": {"0": "PAPER", "1": "PLASTIC", "2": "GLASS"},
    "nc": 3,
}
meta_bytes = json.dumps(metadata).encode("utf-8")

with open(ENGINE_PATH, "wb") as f:
    f.write(len(meta_bytes).to_bytes(4, byteorder="little"))
    f.write(meta_bytes)
    f.write(engine_bytes)

print(f"[OK] metadata={len(meta_bytes)}B + engine={len(engine_bytes)//1024//1024}MB")
print(f"[OK] File size: {Path(ENGINE_PATH).stat().st_size/1e6:.1f} MB")

# Verify
with open(ENGINE_PATH, "rb") as f:
    vlen = int.from_bytes(f.read(4), byteorder="little")
    vmeta = json.loads(f.read(vlen).decode("utf-8"))
print(f"[Verify] {vmeta}")


  Found existing metadata (149B), stripping...
[OK] metadata=163B + engine=18MB
[OK] File size: 19.1 MB
[Verify] {'description': 'YOLOv8 QAT INT8', 'stride': 32, 'task': 'detect', 'batch': 1, 'imgsz': [640, 640], 'names': {'0': 'PAPER', '1': 'PLASTIC', '2': 'GLASS'}, 'nc': 3}


In [2]:
from ultralytics import YOLO
import torch, time, os

DATA_YAML      = "/content/data.yaml"
FP32_WEIGHT    = "/content/drive/MyDrive/QAT/phase_C.pt"
INT8_ENGINE    = "/content/drive/MyDrive/QAT/yolov8-QAT/qat/entropy_skip_detect/weights/best.tensorrt.engine"
WARMUP_RUNS    = 10   # so lan warm up truoc khi do
MEASURE_RUNS   = 100  # so lan do latency
IMGSZ          = 640

def measure_latency(model, imgsz=640, warmup=10, runs=100):
    """Do latency trung binh (ms/image) tren GPU."""
    dummy = torch.zeros(1, 3, imgsz, imgsz).cuda()
    # Warm up
    for _ in range(warmup):
        model(dummy, verbose=False)
    # Measure
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        model(dummy, verbose=False)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / runs * 1000  # ms
    return elapsed

results_table = []

# ── FP32 model goc ─────────────────────────────────────────────────
print("=" * 55)
print("[FP32] Dang danh gia model goc...")
model_fp32 = YOLO(FP32_WEIGHT)

val_fp32 = model_fp32.val(data=DATA_YAML, imgsz=IMGSZ, device=0, verbose=False)
map50_fp32    = val_fp32.box.map50
map5095_fp32  = val_fp32.box.map
lat_fp32      = measure_latency(model_fp32, IMGSZ, WARMUP_RUNS, MEASURE_RUNS)
size_fp32_mb  = os.path.getsize(FP32_WEIGHT) / 1e6

results_table.append({
    "Model":       "FP32 (goc .pt)",
    "mAP50":       map50_fp32,
    "mAP50-95":    map5095_fp32,
    "Latency(ms)": lat_fp32,
    "FPS":         1000 / lat_fp32,
    "Size(MB)":    size_fp32_mb,
})
print(f"  mAP50      : {map50_fp32:.4f}")
print(f"  mAP50-95   : {map5095_fp32:.4f}")
print(f"  Latency    : {lat_fp32:.2f} ms/img")
print(f"  FPS        : {1000/lat_fp32:.1f}")
print(f"  Size       : {size_fp32_mb:.1f} MB")

# ── INT8 engine ────────────────────────────────────────────────────
print("=" * 55)
print("[INT8] Dang danh gia TensorRT INT8 engine...")
model_int8 = YOLO(INT8_ENGINE, task="detect")

val_int8 = model_int8.val(data=DATA_YAML, imgsz=IMGSZ, device=0, verbose=False)
map50_int8    = val_int8.box.map50
map5095_int8  = val_int8.box.map
lat_int8      = measure_latency(model_int8, IMGSZ, WARMUP_RUNS, MEASURE_RUNS)
size_int8_mb  = os.path.getsize(INT8_ENGINE) / 1e6

results_table.append({
    "Model":       "INT8 (.engine)",
    "mAP50":       map50_int8,
    "mAP50-95":    map5095_int8,
    "Latency(ms)": lat_int8,
    "FPS":         1000 / lat_int8,
    "Size(MB)":    size_int8_mb,
})
print(f"  mAP50      : {map50_int8:.4f}")
print(f"  mAP50-95   : {map5095_int8:.4f}")
print(f"  Latency    : {lat_int8:.2f} ms/img")
print(f"  FPS        : {1000/lat_int8:.1f}")
print(f"  Size       : {size_int8_mb:.1f} MB")


[FP32] Dang danh gia model goc...
Ultralytics YOLOv8.0.228 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Model summary (fused): 169 layers, 11126745 parameters, 0 gradients, 28.4 GFLOPs


val: Scanning /content/drive/MyDrive/yolov8/dataset/val/labels.cache... 2434 images, 1 backgrounds, 0 corrupt: 100%|██████████| 2434/2434 [00:00<?, ?it/s]

val: WARNING ⚠️ /content/drive/MyDrive/yolov8/dataset/val/images/2c8035ea9adc.jpg: 1 duplicate labels removed



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 153/153 [00:30<00:00,  4.97it/s]


                   all       2434       3349      0.888      0.842       0.89      0.786
Speed: 0.2ms preprocess, 1.7ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/drive/MyDrive/QAT/yolov8-QAT/runs/detect/val13
  mAP50      : 0.8903
  mAP50-95   : 0.7865
  Latency    : 9.21 ms/img
  FPS        : 108.6
  Size       : 22.5 MB
[INT8] Dang danh gia TensorRT INT8 engine...
Ultralytics YOLOv8.0.228 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Loading /content/drive/MyDrive/QAT/yolov8-QAT/qat/entropy_skip_detect/weights/best.tensorrt.engine for TensorRT inference...


val: Scanning /content/drive/MyDrive/yolov8/dataset/val/labels.cache... 2434 images, 1 backgrounds, 0 corrupt: 100%|██████████| 2434/2434 [00:00<?, ?it/s]

val: WARNING ⚠️ /content/drive/MyDrive/yolov8/dataset/val/images/2c8035ea9adc.jpg: 1 duplicate labels removed



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2434/2434 [00:22<00:00, 109.97it/s]


                   all       2434       3349       0.88      0.844      0.886      0.788
Speed: 0.3ms preprocess, 1.7ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/drive/MyDrive/QAT/yolov8-QAT/runs/detect/val14
Loading /content/drive/MyDrive/QAT/yolov8-QAT/qat/entropy_skip_detect/weights/best.tensorrt.engine for TensorRT inference...
  mAP50      : 0.8857
  mAP50-95   : 0.7876
  Latency    : 3.96 ms/img
  FPS        : 252.7
  Size       : 19.1 MB


In [3]:
# ── In bang so sanh ────────────────────────────────────────────────
fp  = results_table[0]
i8  = results_table[1]

map50_drop  = (fp["mAP50"]    - i8["mAP50"])    / fp["mAP50"]    * 100
map95_drop  = (fp["mAP50-95"] - i8["mAP50-95"]) / fp["mAP50-95"] * 100
speedup     = fp["Latency(ms)"] / i8["Latency(ms)"]
size_ratio  = fp["Size(MB)"]    / i8["Size(MB)"]

print()
print("=" * 60)
print(f"  {"Metric":<18} {"FP32":>10} {"INT8":>10} {"Delta":>10}")
print("-" * 60)
print(f"  {"mAP50":<18} {fp["mAP50"]:>10.4f} {i8["mAP50"]:>10.4f} {-map50_drop:>+9.2f}%")
print(f"  {"mAP50-95":<18} {fp["mAP50-95"]:>10.4f} {i8["mAP50-95"]:>10.4f} {-map95_drop:>+9.2f}%")
print(f"  {"Latency (ms/img)":<18} {fp["Latency(ms)"]:>10.2f} {i8["Latency(ms)"]:>10.2f} {speedup:>+9.2f}x")
print(f"  {"FPS":<18} {fp["FPS"]:>10.1f} {i8["FPS"]:>10.1f}")
print(f"  {"Size (MB)":<18} {fp["Size(MB)"]:>10.1f} {i8["Size(MB)"]:>10.1f} {size_ratio:>+9.2f}x")
print("=" * 60)
print(f"  Ket luan: INT8 nhanh hon {speedup:.2f}x, mAP50 drop {map50_drop:.2f}%")



  Metric                   FP32       INT8      Delta
------------------------------------------------------------
  mAP50                  0.8903     0.8857     -0.53%
  mAP50-95               0.7865     0.7876     +0.15%
  Latency (ms/img)         9.21       3.96     +2.33x
  FPS                     108.6      252.7
  Size (MB)                22.5       19.1     +1.18x
  Ket luan: INT8 nhanh hon 2.33x, mAP50 drop 0.53%


## Buoc 8: Test inference voi INT8 Engine

In [18]:
from ultralytics import YOLO
import glob
import time
import matplotlib.pyplot as plt
import cv2
import os

engine_path = "/content/drive/MyDrive/QAT/yolov8-QAT/qat/entropy_skip_detect/weights/best.tensorrt.engine"
model = YOLO(engine_path, task="detect")

test_images_dir = "/content/drive/MyDrive/QAT/test_imgs"

# Lấy danh sách toàn bộ ảnh (đã bỏ giới hạn [:10])
extensions = ['*.jpg', '*.png', '*.webp', '*.jpeg']
images = []
for ext in extensions:
    images.extend(glob.glob(os.path.join(test_images_dir, ext)))
images = sorted(images)

assert len(images) > 0, f"Khong tim thay anh trong {test_images_dir}"

# Warm up
model(images[0], verbose=False)

# Bắt đầu tính thời gian suy luận cho toàn bộ danh sách
start = time.time()
results = []
for img in images:
    results.extend(model(img, verbose=False))
elapsed = time.time() - start

print(f"[OK] Đã xử lý {len(images)} ảnh: {elapsed:.3f}s | Trung bình: {elapsed/len(images)*1000:.1f} ms/anh")

for i, r in enumerate(results):
    im_bgr = r.plot()
    im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(im_rgb)
    plt.title(f"Image {i+1}: {os.path.basename(images[i])}")
    plt.axis('off')
    plt.show()

Output hidden; open in https://colab.research.google.com to view.